# Bremsstrahlung Models in `espm`

1. **Lifshin Model** (`problem_type="bremsstrahlung"`):
   Two-parameter continuum model ($b_0, b_1$) based on modified Lifshin parametrization ($n_{\text{bkgd}} = 2$).
2. **Power-Law Model** (`problem_type="power_law"`):
   Thin-foil power-law model $N(E, \alpha) = A(E) D(E) \frac{\max(E_0 - E, 0)}{E^\alpha}$ for $\alpha \in [\alpha_{\min}, \alpha_{\max}]$ ($n_{\text{bkgd}} = \text{order} + 1$).

### Approximation Methods for Power-Law Continuum:
- **`taylor`**: Taylor expansion around $\alpha = \alpha_{\max}$ in powers of $\ln(E/E_{\text{ref}})$.
- **`equidistant`**: Exponent sampling linearly spaced across $[\alpha_{\min}, \alpha_{\max}]$.
- **`chebyshev`**: Exponent sampling at Chebyshev-Gauss nodes across $[\alpha_{\min}, \alpha_{\max}]$.


In [ ]:
%matplotlib qt

import copy

import hyperspy.api as hs
import matplotlib.pyplot as plt

from espm.estimators import SmoothNMF

In [ ]:
FILENAME = "../playground/X3-13MAY22_MAP06.bcf"
BIN = 32

signals = hs.load(FILENAME)
signal = signals[1].rebin(scale=(BIN, BIN, 1)).isig[0.2:]
signal.set_signal_type("EDS_espm")

signal.set_analysis_parameters(
    thickness=10e-5,
    density=4.1,
    detector_type="SDD_efficiency.txt",
    width_slope=0.01,
    width_intercept=0.065,
    geom_eff=None,
    xray_db="200keV_xrays.json",
)
signal.change_dtype("float64")

signal.metadata.Sample.elements.append("Ga")
signal.metadata.Sample.elements.append("Cl")

elements = signal.metadata.Sample.elements
print("Signal shape (navigation x signal):", signal.data.shape)
print("Sample elements:", elements)

## 2. Visualizing Basis Continuum Columns for Power-Law Approximations

In [ ]:
methods = ["taylor", "equidistant", "chebyshev"]
method_labels = {
    "taylor": "Taylor Expansion",
    "equidistant": "Equidistant Exponent Sampling",
    "chebyshev": "Chebyshev Nodes Exponent Sampling",
}

order = 3
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)

for idx, method in enumerate(methods):
    s_method = copy.deepcopy(signal)
    s_method.build_G(problem_type="power_law", order=order, alpha_min=1.0, alpha_max=2.0, method=method)
    G_bkgd = s_method.G_[:, -s_method.model.n_bkgd:]
    
    ax = axes[idx]
    energy = s_method.energy_axis
    for col in range(G_bkgd.shape[1]):
        ax.plot(energy, G_bkgd[:, col], label=f"Basis Col {col+1}")
    ax.set_title(method_labels[method], fontsize=11, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(fontsize=9)
    ax.set_xlabel("Energy (keV)")
    if idx == 0:
        ax.set_ylabel("Intensity (a.u.)")

plt.suptitle("Power-Law Bremsstrahlung Basis Column Shapes (Order = 3, 4 Basis Columns)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 3. Direct Tensor Decomposition via `signal.decomposition()`

We use `signal.decomposition()` directly on deep copies of `signal` (`s.decomposition(algorithm=SmoothNMF(n_components=2, G=s.model, hspy_comp=True))`):


In [ ]:
n_components = 2
energy = signal.energy_axis
avg_spectrum = signal.data.mean(axis=(0, 1))

# Lifshin Model decomposition
s_lifshin = copy.deepcopy(signal)
s_lifshin.build_G(problem_type="bremsstrahlung")
decomp_lif = SmoothNMF(n_components=n_components, G=s_lifshin.G, hspy_comp=True, max_iter=500, tol=1e-5, verbose=0)
s_lifshin.decomposition(algorithm=decomp_lif)
W_lif, G_lif, H_lif = decomp_lif.W_, decomp_lif.G_, decomp_lif.H_
n_bkgd_lif = s_lifshin.model.n_bkgd
bkgd_lifshin_avg = (G_lif @ W_lif @ H_lif.mean(axis=1)).ravel()

# Power-Law Models decomposition
bkgds_power_law_avg = {}
fits_power_law_2d = {}

for method in methods:
    s_method = copy.deepcopy(signal)
    s_method.build_G(problem_type="power_law", order=order, alpha_min=1.0, alpha_max=2.0, method=method)
    alg = SmoothNMF(n_components=n_components, G=s_method.G, hspy_comp=True, max_iter=500, tol=1e-5, verbose=0)
    s_method.decomposition(algorithm=alg)
    W_pl, G_pl, H_pl = alg.W_, alg.G_, alg.H_
    n_bkgd_pl = s_method.model.n_bkgd
    fits_power_law_2d[method] = G_pl @ W_pl @ H_pl
    bkgds_power_law_avg[method] = (G_pl @ W_pl @ H_pl.mean(axis=1)).ravel()

# Plot spatially averaged continuum background extracted from signal.decomposition
plt.figure(figsize=(12, 6))
plt.plot(energy, avg_spectrum, color="gray", alpha=0.5, label="Experimental Average Spectrum")
plt.plot(energy, bkgd_lifshin_avg, color="black", linestyle="--", label="Lifshin Background (signal.decomposition)", linewidth=2)

for method in methods:
    plt.plot(energy, bkgds_power_law_avg[method], label=f"Power-Law ({method}) Background (signal.decomposition)", linewidth=1.5)

plt.xlabel("Energy (keV)")
plt.ylabel("Intensity (counts)")
plt.title("Extracted Bremsstrahlung Continuum (Spatially Averaged from signal.decomposition)", fontsize=12, fontweight="bold")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()